In [ ]:
# =============================================================================
# Step 0: GEE Authentication + Drive mount 
# =============================================================================

# ── Mount Google Drive (for cache + model persistence) ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Install/upgrade earthengine-api if needed ─────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', '--upgrade', 'earthengine-api'],
               check=True)


In [ ]:
# ── Authenticate and initialize GEE ───────────────────────────────────────────
import ee

try:
    ee.Initialize(project='ru-thesis-2026')
    print("GEE already authenticated ✓")
except Exception:
    print("Running GEE authentication flow...")
    ee.Authenticate()
    ee.Initialize(project='ru-thesis-2026')
    print("GEE initialized ✓")

# ── Verify connection ──────────────────────────────────────────────────────────
test = ee.Number(42).getInfo()
assert test == 42, "GEE connection test failed"
print(f"GEE connection verified ✓  (test value: {test})")


In [ ]:
# ── Set cache + model dirs on Drive so they survive session resets ─────────────
import os
CACHE_DIR  = '/content/drive/MyDrive/RU_Thesis_2026/gee_cache'
MODELS_DIR = '/content/drive/MyDrive/RU_Thesis_2026/models'
FIGURES_DIR = '/content/drive/MyDrive/RU_Thesis_2026/figures'

os.makedirs(CACHE_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"Cache dir  : {CACHE_DIR}")
print(f"Models dir : {MODELS_DIR}")
print(f"Figures dir: {FIGURES_DIR}")

In [ ]:
# =============================================================================
# Copy .npy cache from Google Drive → local Colab SSD
# =============================================================================
import shutil, glob, os, time

DRIVE_CACHE = CACHE_DIR                   
LOCAL_CACHE = '/content/npy_cache'        
os.makedirs(LOCAL_CACHE, exist_ok=True)

# Count what's already on local disk — avoids re-copying if cell re-run
already_local = glob.glob(os.path.join(LOCAL_CACHE, '*.npy'))
drive_files = (glob.glob(os.path.join(DRIVE_CACHE, '*.npy')) +
               glob.glob(os.path.join(DRIVE_CACHE, '*.pkl')))

print(f"Drive cache : {len(drive_files):,} .npy files")
print(f"Local cache : {len(already_local):,} .npy files already copied")

if len(already_local) >= len(drive_files):
    print("Local cache is up to date — skipping copy ✓")
else:
    to_copy = [f for f in drive_files
               if not os.path.exists(
                   os.path.join(LOCAL_CACHE, os.path.basename(f)))]
    print(f"Copying {len(to_copy):,} files to local SSD...")

    t0 = time.time()
    for i, f in enumerate(to_copy):
        shutil.copy(f, LOCAL_CACHE)
        if (i + 1) % 500 == 0:
            elapsed = time.time() - t0
            rate    = (i + 1) / elapsed
            remaining = (len(to_copy) - i - 1) / rate
            print(f"  {i+1:,}/{len(to_copy):,} files "
                  f"— {rate:.0f} files/sec "
                  f"— ~{remaining/60:.1f} min remaining")

    elapsed = time.time() - t0
    print(f"\n  Done in {elapsed/60:.1f} min ✓")

# ── Override CACHE_DIR to point to fast local copy ────────────────────────────
CACHE_DIR = LOCAL_CACHE
print(f"\nCACHE_DIR now points to local SSD: {CACHE_DIR}")
print("All subsequent cells will read from local SSD automatically ✓")

In [ ]:
# =============================================================================
# Step 1: Imports + config
# =============================================================================
import numpy as np
import pandas as pd
import os
import ee
import joblib
import calendar
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                             balanced_accuracy_score,
                             recall_score, precision_score)
import matplotlib.pyplot as plt

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")

# ── GEE asset paths ───────────────────────────────────────────────────────────
GEE_PROJECT   = 'ru-thesis-2026'
DYNAMIC_ASSET = f'projects/{GEE_PROJECT}/assets/Ontario_Monthly_2010-2024'
STATIC_ASSET  = (f'projects/{GEE_PROJECT}/assets/'
                 f'Ontario_Static_2010-2024/Static_2010')

# ── Band names ────────────────────────────────────────────────────────────────
DYNAMIC_BANDS = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value', 'target_ignition'
]
STATIC_BANDS = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
TARGET_BAND  = 'target_ignition'

# ── Year splits ───────────────────────────────────────────────────────────────
TRAIN_YEARS = list(range(2010, 2021))
VAL_YEARS   = [2021, 2022]

# ── ConvLSTM config ───────────────────────────────────────────────────────────
SEQ_LEN    = 14      # days of lookback history
BATCH_SIZE = 1      # minimum — keeps RAM flat
EPOCHS     = 50     # early stopping will cut short
PATIENCE   = 8      # epochs without val_auc improvement before stopping
POS_WEIGHT = 500.0  # fire pixel loss weight — tuned for 3295:1 imbalance
FILTERS    = 16     # ConvLSTM filter count — reduced from 32 to save memory
DROPOUT    = 0.4    # dropout rate

# ── GEE retry config ──────────────────────────────────────────────────────────
GEE_MAX_RETRIES = 5
GEE_RETRY_BASE  = 2.0   # exponential backoff base in seconds

In [ ]:
# =============================================================================
# Step 2 — Ontario bbox + grid dimensions
# =============================================================================
print("="*65)
print("STEP 2: Defining Ontario region")
print("="*65)

ONTARIO_BBOX = ee.Geometry.Rectangle([-95.15, 41.67, -74.32, 56.85])

sample_asset = ee.Image(f'{DYNAMIC_ASSET}/Month_201001')
sample_proj  = sample_asset.projection()
sample_info  = sample_proj.getInfo()
print(f"  Asset CRS  : {sample_info.get('crs', 'N/A')}")
print(f"  Asset scale: {sample_proj.nominalScale().getInfo():.0f} m")

In [ ]:
# =============================================================================
# Step 3 — Static features
# =============================================================================
print("="*65)
print("STEP 3: Loading static features")
print("="*65)

static_cache = os.path.join(CACHE_DIR, 'static_array.npy')
static_meta  = os.path.join(CACHE_DIR, 'static_meta.pkl')

# ── If .pkl missing from local SSD, copy it from Drive ───────────────────────
if not os.path.exists(static_meta) and CACHE_DIR == LOCAL_CACHE:
    drive_meta = static_meta.replace(LOCAL_CACHE, DRIVE_CACHE)
    if os.path.exists(drive_meta):
        import shutil
        shutil.copy(drive_meta, static_meta)
        print("  Copied static_meta.pkl from Drive to local SSD ✓")

if os.path.exists(static_cache) and os.path.exists(static_meta):
    print("  Loading cached static array...")
    static_arr = np.load(static_cache)
    meta       = joblib.load(static_meta)
else:
    print("  Downloading static features from GEE...")
    static_img  = ee.Image(STATIC_ASSET).select(STATIC_BANDS)
    static_data = static_img.sampleRectangle(
        region=ONTARIO_BBOX, defaultValue=0,
        properties=STATIC_BANDS).getInfo()

    static_arrays = []
    for band in STATIC_BANDS:
        arr = np.array(static_data['properties'][band], dtype=np.float32)
        static_arrays.append(arr)
        print(f"    {band}: {arr.shape}")

    H, W       = static_arrays[0].shape
    static_arr = np.stack(static_arrays, axis=-1)
    meta       = {'H': H, 'W': W,
                  'static_bands': STATIC_BANDS,
                  'dynamic_bands': DYNAMIC_BANDS}

    np.save(static_cache, static_arr)
    joblib.dump(meta, static_meta)

    # Mirror .pkl to Drive too
    if CACHE_DIR == LOCAL_CACHE:
        joblib.dump(meta, static_meta.replace(LOCAL_CACHE, DRIVE_CACHE))

H, W       = static_arr.shape[:2]
N_STATIC   = static_arr.shape[2]
N_DYNAMIC  = len(DYNAMIC_BANDS) - 1
N_FEATURES = N_DYNAMIC + N_STATIC

print(f"  Static array shape : {static_arr.shape}")
print(f"  Grid : H={H}  W={W}")
print(f"  Dynamic features : {N_DYNAMIC}")
print(f"  Static features  : {N_STATIC}")
print(f"  Total features   : {N_FEATURES}")

In [ ]:
# =============================================================================
# Step 4 — GEE fetch helpers + download to disk cache
# =============================================================================
print("="*65)
print("STEP 4: Downloading daily grids to disk cache")
print("="*65)

def gee_fetch_day(img, all_bands, feature_base_names, date_str, H, W):
    """
    Fetch one day's grids from a monthly GEE image with exponential backoff.
    Returns (feat_grid (H,W,N_DYN), tgt_grid (H,W,1), success: bool)
    """
    day_feat_names = [f'{b}_{date_str}' for b in feature_base_names]
    day_tgt_name   = f'{TARGET_BAND}_{date_str}'

    missing = [b for b in day_feat_names + [day_tgt_name]
               if b not in all_bands]
    if missing:
        print(f"    SKIP {date_str}: {len(missing)} bands missing")
        return (np.zeros((H, W, len(feature_base_names)), dtype=np.float32),
                np.zeros((H, W, 1), dtype=np.float32), False)

    feat_img = img.select(day_feat_names).rename(feature_base_names)
    tgt_img  = img.select([day_tgt_name]).rename([TARGET_BAND])
    last_err = None

    for attempt in range(1, GEE_MAX_RETRIES + 1):
        try:
            feat_data = feat_img.sampleRectangle(
                region=ONTARIO_BBOX, defaultValue=0,
                properties=feature_base_names).getInfo()
            tgt_data  = tgt_img.sampleRectangle(
                region=ONTARIO_BBOX, defaultValue=0,
                properties=[TARGET_BAND]).getInfo()

            feat_arrays = []
            for band in feature_base_names:
                arr = np.array(feat_data['properties'][band], dtype=np.float32)
                arr = arr[:H, :W]
                if arr.shape != (H, W):
                    padded = np.zeros((H, W), dtype=np.float32)
                    padded[:arr.shape[0], :arr.shape[1]] = arr
                    arr = padded
                feat_arrays.append(arr)
            feat_grid = np.stack(feat_arrays, axis=-1)

            tgt_arr = np.array(
                tgt_data['properties'][TARGET_BAND], dtype=np.float32)
            tgt_arr = tgt_arr[:H, :W]
            if tgt_arr.shape != (H, W):
                padded = np.zeros((H, W), dtype=np.float32)
                padded[:tgt_arr.shape[0], :tgt_arr.shape[1]] = tgt_arr
                tgt_arr = padded
            tgt_grid = tgt_arr[:, :, np.newaxis]

            return feat_grid, tgt_grid, True

        except Exception as e:
            last_err = e
            wait = GEE_RETRY_BASE ** attempt
            print(f"    RETRY {attempt}/{GEE_MAX_RETRIES} — {date_str} "
                  f"({type(e).__name__}: {str(e)[:60]}) — waiting {wait:.0f}s")
            time.sleep(wait)

    print(f"    FAILED {date_str} after {GEE_MAX_RETRIES} attempts")
    return (np.zeros((H, W, len(feature_base_names)), dtype=np.float32),
            np.zeros((H, W, 1), dtype=np.float32), False)


def download_all_months_to_cache(years):
    total_days, total_missing = 0, 0
    for year in years:
        for month in range(1, 13):
            n_days     = calendar.monthrange(year, month)[1]
            asset_name = f'{DYNAMIC_ASSET}/Month_{year}{month:02d}'

            cached = [d for d in range(1, n_days + 1)
                      if os.path.exists(os.path.join(
                          CACHE_DIR, f'feat_{year}{month:02d}{d:02d}.npy'))]
            if len(cached) == n_days:
                print(f"  {year}-{month:02d}: all {n_days} days cached ✓")
                total_days += n_days
                continue

            try:
                img       = ee.Image(asset_name)
                all_bands = img.bandNames().getInfo()
            except Exception as e:
                print(f"  WARNING: could not open {asset_name}: {e}")
                total_missing += n_days
                continue

            feature_base_names = DYNAMIC_BANDS[:-1]
            month_fires = 0

            for day in range(1, n_days + 1):
                date_str   = f'{year}{month:02d}{day:02d}'
                cache_feat = os.path.join(CACHE_DIR, f'feat_{date_str}.npy')
                cache_tgt  = os.path.join(CACHE_DIR, f'tgt_{date_str}.npy')

                if os.path.exists(cache_feat) and os.path.exists(cache_tgt):
                    total_days += 1
                    continue

                feat_grid, tgt_grid, success = gee_fetch_day(
                    img, all_bands, feature_base_names, date_str, H, W)

                np.save(cache_feat, feat_grid)
                np.save(cache_tgt,  tgt_grid)

                # Mirror to Drive so cache survives session resets
                if CACHE_DIR == LOCAL_CACHE:
                    np.save(cache_feat.replace(LOCAL_CACHE, DRIVE_CACHE),
                            feat_grid)
                    np.save(cache_tgt.replace(LOCAL_CACHE, DRIVE_CACHE),
                            tgt_grid)

                if success and tgt_grid.sum() > 0:
                    month_fires += 1
                elif not success:
                    total_missing += 1

                total_days += 1
                del feat_grid, tgt_grid

            print(f"  {year}-{month:02d}: {n_days} days done "
                  f"({month_fires} fire days)")

    print(f"\n  Total days processed : {total_days}")
    print(f"  Total failed/missing : {total_missing}")


download_all_months_to_cache(TRAIN_YEARS + VAL_YEARS)

In [ ]:
# =============================================================================
# Steps 5 - 7: Guard: ensure ontario_mask exists before proceeding
# Rebuilds from cache or GEE if kernel was restarted
# =============================================================================
import datetime

if 'ontario_mask' not in dir() or ontario_mask is None:
    print("  ontario_mask not found in memory — rebuilding...")
    mask_cache = os.path.join(CACHE_DIR, 'ontario_mask.npy')

    if os.path.exists(mask_cache):
        ontario_mask = np.load(mask_cache).astype(bool)
        print(f"  Loaded from cache ✓  ({ontario_mask.sum():,} Ontario pixels)")

    else:
        # Also check Drive if on local SSD
        drive_mask = mask_cache.replace(LOCAL_CACHE, DRIVE_CACHE) \
                     if CACHE_DIR == LOCAL_CACHE else mask_cache
        if os.path.exists(drive_mask):
            import shutil
            shutil.copy(drive_mask, mask_cache)
            ontario_mask = np.load(mask_cache).astype(bool)
            print(f"  Copied from Drive + loaded ✓  "
                  f"({ontario_mask.sum():,} Ontario pixels)")

        else:
            print("  Not on disk — fetching province_id from GEE...")
            sample_img   = ee.Image(f'{DYNAMIC_ASSET}/Month_201005')
            province_data = sample_img.select(['province_id']).sampleRectangle(
                region=ONTARIO_BBOX, defaultValue=0,
                properties=['province_id']).getInfo()
            province_arr = np.array(
                province_data['properties']['province_id'],
                dtype=np.float32)[:H, :W]
            ontario_mask = (province_arr > 0)
            np.save(mask_cache, ontario_mask)
            if CACHE_DIR == LOCAL_CACHE:
                np.save(drive_mask, ontario_mask)
            print(f"  Built from GEE + cached ✓  "
                  f"({ontario_mask.sum():,} Ontario pixels)")
else:
    print(f"  ontario_mask already in memory ✓  "
          f"({ontario_mask.sum():,} Ontario pixels)")

# Also ensure feat_mean / feat_std are available
if 'feat_mean' not in dir() or feat_mean is None:
    scaler_cache = os.path.join(CACHE_DIR, 'feature_scaler.pkl')
    if os.path.exists(scaler_cache):
        scaler    = joblib.load(scaler_cache)
        feat_mean = scaler['mean']
        feat_std  = scaler['std']
        print(f"  feat_mean/std loaded from cache ✓")
    else:
        raise RuntimeError(
            "feature_scaler.pkl not found — run Cell 5 Step 6 first "
            "to compute normalisation statistics.")

# Also ensure static_arr is available
if 'static_arr' not in dir() or static_arr is None:
    static_cache = os.path.join(CACHE_DIR, 'static_array.npy')
    if os.path.exists(static_cache):
        static_arr = np.load(static_cache)
        H, W       = static_arr.shape[:2]
        N_STATIC   = static_arr.shape[2]
        N_DYNAMIC  = len(DYNAMIC_BANDS) - 1
        N_FEATURES = N_DYNAMIC + N_STATIC
        print(f"  static_arr loaded from cache ✓  shape={static_arr.shape}")
    else:
        raise RuntimeError(
            "static_array.npy not found — run Cell 3 first.")

print(f"\n  All dependencies ready ✓")
print(f"  Grid         : H={H}  W={W}")
print(f"  Ontario px   : {ontario_mask.sum():,}")
print(f"  N_FEATURES   : {N_FEATURES}")


print("="*65)
print("STEP 5: Building day key lists")
print("="*65)

# ── Seasonal filter matched to XGBoost DOY_START_HARD / DOY_END_HARD ─────────
DOY_START_HARD = 91    # April 1
DOY_END_HARD   = 310   # November 6

def is_fire_season(year, month, day):
    """
    Returns True if date falls within DOY 91–310.
    Matches XGBoost pipeline exactly.
    """
    import datetime
    doy = datetime.date(year, month, day).timetuple().tm_yday
    return DOY_START_HARD <= doy <= DOY_END_HARD

def get_ordered_keys(years):
    """
    Returns sorted (year, month, day) tuples for days that:
    1. Fall within fire season (DOY 91-310)
    2. Exist on disk in cache
    """
    keys = []
    for year in years:
        for month in range(1, 13):
            n_days = calendar.monthrange(year, month)[1]
            for day in range(1, n_days + 1):
                if not is_fire_season(year, month, day):
                    continue
                date_str = f'{year}{month:02d}{day:02d}'
                if os.path.exists(
                        os.path.join(CACHE_DIR, f'feat_{date_str}.npy')):
                    keys.append((year, month, day))
    return keys

train_keys = get_ordered_keys(TRAIN_YEARS)
val_keys   = get_ordered_keys(VAL_YEARS)
print(f"  Train days (DOY {DOY_START_HARD}–{DOY_END_HARD}) : {len(train_keys)}")
print(f"  Val days   (DOY {DOY_START_HARD}–{DOY_END_HARD}) : {len(val_keys)}")

# Verify pixel count matches XGBoost scale
expected_pixels_per_day = ontario_mask.sum()
print(f"\n  Ontario pixels/day : {expected_pixels_per_day:,}")
print(f"  Expected val total : "
      f"{expected_pixels_per_day * len(val_keys):,} pixels")
print(f"  XGBoost val total  : ~5,221,000 pixels")
print(f"  Match: "
      f"{'✓' if abs(expected_pixels_per_day * len(val_keys) - 5_221_000) < 500_000 else '✗ — check ontario_mask'}")

# ── Step 6: Normalisation ──────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 6: Computing normalisation statistics")
print("="*65)

scaler_cache = os.path.join(CACHE_DIR, 'feature_scaler.pkl')

if os.path.exists(scaler_cache):
    scaler    = joblib.load(scaler_cache)
    feat_mean = scaler['mean']
    feat_std  = scaler['std']
    print("  Loaded cached scaler ✓")
else:
    print("  Streaming training days from disk (Welford's algorithm)...")
    n_pixels = 0
    mean_acc = np.zeros(N_FEATURES, dtype=np.float64)
    M2_acc   = np.zeros(N_FEATURES, dtype=np.float64)

    for i, (year, month, day) in enumerate(train_keys):
        date_str = f'{year}{month:02d}{day:02d}'
        feat_dyn = np.load(os.path.join(CACHE_DIR, f'feat_{date_str}.npy'))
        full     = np.concatenate([feat_dyn, static_arr], axis=-1)
        del feat_dyn
        pixels = full.reshape(-1, N_FEATURES).astype(np.float64)
        del full
        for px in pixels:
            n_pixels += 1
            delta     = px - mean_acc
            mean_acc += delta / n_pixels
            M2_acc   += delta * (px - mean_acc)
        del pixels
        if (i + 1) % 100 == 0:
            print(f"    {i+1}/{len(train_keys)} days...")

    feat_mean = mean_acc.astype(np.float32)
    feat_std  = np.sqrt(M2_acc / n_pixels).astype(np.float32)
    feat_std  = np.where(feat_std == 0, 1.0, feat_std)
    joblib.dump({'mean': feat_mean, 'std': feat_std}, scaler_cache)
    print(f"  Scaler computed on {n_pixels:,} pixels ✓")

# ── Step 7: tf.data pipeline ───────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 7: Building disk-based tf.data pipeline")
print("="*65)

# Load fire_cause arrays — needed for pixel-level filtering
# fire_cause is stored in the dynamic asset but not in feat grids
# Separate cache for it
FIRE_CAUSE_BANDS = ['fire_cause']

def load_fire_cause_day(year, month, day):
    """
    Load fire_cause grid for one day.
    Returns (H, W) int array. Cached to disk like feat/tgt.
    """
    date_str  = f'{year}{month:02d}{day:02d}'
    cache_fc  = os.path.join(CACHE_DIR, f'fc_{date_str}.npy')

    if os.path.exists(cache_fc):
        return np.load(cache_fc)

    # Not cached — fetch from GEE
    asset_name = f'{DYNAMIC_ASSET}/Month_{year}{month:02d}'
    try:
        img       = ee.Image(asset_name)
        band_name = f'fire_cause_{date_str}'
        all_bands = img.bandNames().getInfo()

        if band_name not in all_bands:
            # fire_cause not available — return all-zero array (treat as valid)
            arr = np.zeros((H, W), dtype=np.int8)
        else:
            fc_img  = img.select([band_name]).rename(['fire_cause'])
            fc_data = fc_img.sampleRectangle(
                region=ONTARIO_BBOX, defaultValue=0,
                properties=['fire_cause']).getInfo()
            arr = np.array(
                fc_data['properties']['fire_cause'],
                dtype=np.int8)[:H, :W]
            if arr.shape != (H, W):
                padded = np.zeros((H, W), dtype=np.int8)
                padded[:arr.shape[0], :arr.shape[1]] = arr
                arr = padded

        np.save(cache_fc, arr)
        if CACHE_DIR == LOCAL_CACHE:
            np.save(cache_fc.replace(LOCAL_CACHE, DRIVE_CACHE), arr)
        return arr

    except Exception as e:
        print(f"    WARNING fire_cause {date_str}: {e}")
        return np.zeros((H, W), dtype=np.int8)


def build_valid_pixel_mask(tgt_grid, fire_cause_grid):
    """
    Pixel-level validity mask matching XGBoost clean_and_filter():
      - Ontario pixels only       (ontario_mask)
      - target_ignition in {0,1}  (no NaN/nodata)
      - fire_cause in [0,5]
    Returns boolean (H, W) mask — True = valid pixel.
    """
    valid = ontario_mask.copy()                          # Ontario only
    valid &= np.isfinite(tgt_grid[:, :, 0])             # no NaN targets
    valid &= np.isin(tgt_grid[:, :, 0], [0, 1])         # binary target only
    valid &= (fire_cause_grid >= 0) & (fire_cause_grid <= 5)  # valid cause
    return valid


def disk_sequence_generator(keys, seq_len):
    """
    Yields (X_seq, y) by reading .npy files from disk.
    Applies same pixel filters as XGBoost clean_and_filter().
    Non-valid pixels are zeroed out so model ignores them.
    """
    for i in range(seq_len, len(keys)):
        seq_keys = keys[i - seq_len : i]
        tgt_key  = keys[i]

        # ── Build sequence ─────────────────────────────────────────────────
        frames = []
        for year, month, day in seq_keys:
            date_str = f'{year}{month:02d}{day:02d}'
            feat_dyn = np.load(
                os.path.join(CACHE_DIR, f'feat_{date_str}.npy'))
            full = np.concatenate(
                [feat_dyn, static_arr], axis=-1).astype(np.float32)
            full = (full - feat_mean) / feat_std

            # Zero out non-Ontario pixels
            full[~ontario_mask] = 0.0
            frames.append(full)
            del feat_dyn

        x_seq = np.stack(frames, axis=0)   # (seq_len, H, W, N_FEATURES)
        del frames

        # ── Load target day ────────────────────────────────────────────────
        year, month, day = tgt_key
        date_str  = f'{year}{month:02d}{day:02d}'
        tgt_grid  = np.load(
            os.path.join(CACHE_DIR, f'tgt_{date_str}.npy'))  # (H, W, 1)
        fc_grid   = load_fire_cause_day(year, month, day)     # (H, W)

        # ── Apply pixel validity mask ──────────────────────────────────────
        valid_mask = build_valid_pixel_mask(tgt_grid, fc_grid)

        # Zero out invalid pixels in target
        tgt_out = tgt_grid.copy()
        tgt_out[~valid_mask] = 0.0

        # Zero out invalid pixels in all sequence frames
        x_seq[:, ~valid_mask, :] = 0.0

        yield x_seq.astype(np.float32), tgt_out.astype(np.float32)
        del x_seq, tgt_grid, tgt_out, fc_grid


def make_tf_dataset(keys, seq_len, batch_size, shuffle=False, repeat=True):
    output_signature = (
        tf.TensorSpec(shape=(seq_len, H, W, N_FEATURES), dtype=tf.float32),
        tf.TensorSpec(shape=(H, W, 1),                   dtype=tf.float32),
    )
    ds = tf.data.Dataset.from_generator(
        lambda: disk_sequence_generator(keys, seq_len),
        output_signature=output_signature)
    if shuffle:
        ds = ds.shuffle(buffer_size=50, reshuffle_each_iteration=True)
    if repeat:
        ds = ds.repeat()   # prevents generator exhaustion between epochs
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds    = make_tf_dataset(train_keys, SEQ_LEN, BATCH_SIZE,
                              shuffle=True,  repeat=True)
val_ds      = make_tf_dataset(val_keys,   SEQ_LEN, BATCH_SIZE,
                              shuffle=False, repeat=True)
val_ds_eval = make_tf_dataset(val_keys,   SEQ_LEN, BATCH_SIZE,
                              shuffle=False, repeat=False)  # for Step 9

train_steps = (len(train_keys) - SEQ_LEN) // BATCH_SIZE
val_steps   = (len(val_keys)   - SEQ_LEN) // BATCH_SIZE

print(f"  Train steps/epoch : {train_steps}")
print(f"  Val   steps/epoch : {val_steps}")
print(f"  Peak RAM per step : "
      f"~{SEQ_LEN * H * W * N_FEATURES * 4 / 1e6 * BATCH_SIZE:.0f} MB")

def is_fire_season(year, month, day):
    """
    Returns True if date falls within fire season:
    April 1 – November 5  (inclusive)
    Excludes November 6 – March 31
    """
    if month < 4 or month > 11:
        return False
    if month == 4 and day < 1:
        return False
    if month == 11 and day > 5:
        return False
    return True

def get_ordered_keys(years, fire_season_only=True):
    """Returns sorted (year, month, day) tuples for days that exist on disk."""
    keys = []
    for year in years:
        for month in range(1, 13):
            n_days = calendar.monthrange(year, month)[1]
            for day in range(1, n_days + 1):
                # Apply seasonal filter
                if fire_season_only and not is_fire_season(year, month, day):
                    continue
                date_str = f'{year}{month:02d}{day:02d}'
                if os.path.exists(
                        os.path.join(CACHE_DIR, f'feat_{date_str}.npy')):
                    keys.append((year, month, day))
    return keys

train_keys = get_ordered_keys(TRAIN_YEARS, fire_season_only=True)
val_keys   = get_ordered_keys(VAL_YEARS,   fire_season_only=True)
print(f"  Train days (fire season only) : {len(train_keys)}")
print(f"  Val days   (fire season only) : {len(val_keys)}")

In [ ]:
# =============================================================================
# Step 8: Build/resume + train model
# =============================================================================
print("="*65)
print("STEP 8: Building ConvLSTM2D model")
print("="*65)

def weighted_bce(y_true, y_pred, pos_weight=POS_WEIGHT):
    y_pred  = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    weights = y_true * pos_weight + (1 - y_true)
    bce     = -(y_true * tf.math.log(y_pred) +
                (1 - y_true) * tf.math.log(1 - y_pred))
    return tf.reduce_mean(weights * bce)


def build_convlstm_model(seq_len, H, W, n_features,
                         filters=16, kernel_size=3, dropout=0.4):
    inputs = keras.Input(
        shape=(seq_len, H, W, n_features), name='input_sequence')

    x = layers.ConvLSTM2D(
        filters=filters, kernel_size=kernel_size,
        padding='same', return_sequences=True,
        activation='tanh', name='convlstm_1')(inputs)
    x = layers.BatchNormalization()(x)

    x = layers.ConvLSTM2D(
        filters=filters // 2, kernel_size=kernel_size,
        padding='same', return_sequences=False,
        activation='tanh', name='convlstm_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv2D(
        filters=filters // 2, kernel_size=3,
        padding='same', activation='relu',
        name='spatial_refine')(x)

    outputs = layers.Conv2D(
        filters=1, kernel_size=1,
        padding='same', activation='sigmoid',
        name='output')(x)

    return keras.Model(inputs, outputs, name='ConvLSTM_Ignition')


# ── Build or resume ────────────────────────────────────────────────────────────
INITIAL_EPOCH = 0   # ← set to last completed epoch number to resume

resume_path = os.path.join(MODELS_DIR, 'model_convlstm_best.keras')

if os.path.exists(resume_path) and INITIAL_EPOCH > 0:
    print(f"  Resuming from epoch {INITIAL_EPOCH}...")
    model = keras.models.load_model(
        resume_path, custom_objects={'weighted_bce': weighted_bce})
    print(f"  Loaded: {resume_path} ✓")
else:
    print("  Building new model from scratch...")
    model = build_convlstm_model(SEQ_LEN, H, W, N_FEATURES,
                                 filters=FILTERS, dropout=DROPOUT)

model.summary()

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = weighted_bce,
    metrics   = [keras.metrics.AUC(name='auc'),
                 keras.metrics.Recall(name='recall')]
)

cb_list = [
    callbacks.EarlyStopping(
        monitor='val_auc', patience=PATIENCE,
        mode='max', restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=4, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, 'model_convlstm_best.keras'),
        monitor='val_auc', mode='max',
        save_best_only=True, verbose=1),
    callbacks.CSVLogger(
        os.path.join(MODELS_DIR, 'training_log.csv'),
        append=True),
]

history = model.fit(
    train_ds,
    validation_data  = val_ds,
    epochs           = EPOCHS,
    initial_epoch    = INITIAL_EPOCH,
    steps_per_epoch  = train_steps,
    validation_steps = val_steps,
    callbacks        = cb_list,
    verbose          = 1
)

In [ ]:
# =============================================================================
# Step 9 — Evaluation matched to XGBoost validation set exactly
# =============================================================================
print("="*65)
print("STEP 9: Evaluating on validation set")
print("="*65)

model_best = keras.models.load_model(
    os.path.join(MODELS_DIR, 'model_convlstm_best.keras'),
    custom_objects={'weighted_bce': weighted_bce})

print("  Streaming predictions (Ontario + fire season + valid pixels)...")
y_true_list, y_prob_list = [], []

ontario_flat = ontario_mask.flatten()

for (year, month, day), (x_batch, y_batch) in zip(
        val_keys[SEQ_LEN:], val_ds_eval):

    # Seasonal filter — already guaranteed by get_ordered_keys
    # but double-check for safety
    if not is_fire_season(year, month, day):
        continue

    prob_batch = model_best.predict(x_batch, verbose=0)

    # Load fire_cause for this target day — pixel validity filter
    fc_grid    = load_fire_cause_day(year, month, day)
    tgt_np     = y_batch.numpy()[0, :, :, 0]   # (H, W)

    valid_mask = (
        ontario_mask &
        np.isin(tgt_np, [0, 1]) &
        (fc_grid >= 0) & (fc_grid <= 5)
    ).flatten()

    y_true_list.append(y_batch.numpy().reshape(-1)[valid_mask])
    y_prob_list.append(prob_batch.reshape(-1)[valid_mask])

y_true_flat = np.concatenate(y_true_list)
y_prob_flat = np.concatenate(y_prob_list)

pixels_per_day = valid_mask.sum()   # approximate — varies slightly by day
implied_days   = len(y_true_flat) / ontario_mask.sum()
print(f"  Total pixels       : {len(y_true_flat):,}")
print(f"  Fire pixels        : {int(y_true_flat.sum()):,}")
print(f"  Implied val days   : {implied_days:.0f}")
print(f"  XGBoost val pixels : ~5,221,000")
print(f"  Match              : "
      f"{'✓ comparable' if abs(len(y_true_flat) - 5_221_000) < 1_000_000 else '✗ still differs — check filters'}")

# ── Threshold search ───────────────────────────────────────────────────────────
MIN_RECALL  = 0.70
best_thr, best_prec = 0.5, 0.0
for thr in np.linspace(0.01, 0.99, 200):
    y_t  = (y_prob_flat >= thr).astype(int)
    rec  = recall_score(y_true_flat, y_t, zero_division=0)
    prec = precision_score(y_true_flat, y_t, zero_division=0)
    if rec >= MIN_RECALL and prec > best_prec:
        best_prec, best_thr = prec, thr

y_pred_flat    = (y_prob_flat >= best_thr).astype(int)
auc            = roc_auc_score(y_true_flat, y_prob_flat)
tn, fp, fn, tp = confusion_matrix(y_true_flat, y_pred_flat).ravel()
rec            = tp / (tp + fn)
prec           = tp / (tp + fp) if (tp + fp) > 0 else 0
fpr            = fp / (fp + tn)
bal            = balanced_accuracy_score(y_true_flat, y_pred_flat)

print(f"\n  CONVLSTM2D — VALIDATION RESULTS (matched to XGBoost evaluation)")
print(f"  {'─'*55}")
print(f"  Total pixels       : {len(y_true_flat):,}  "
      f"(XGBoost: ~5,221,000)")
print(f"  Fire pixels        : {int(y_true_flat.sum()):,}  "
      f"(XGBoost: check TP+FN)")
print(f"  {'─'*55}")
print(f"  ROC-AUC            : {auc:.4f}")
print(f"  Balanced accuracy  : {bal:.4f}")
print(f"  Recall             : {rec:.4f}  ({tp} of {tp+fn} fire pixels)")
print(f"  Precision          : {prec:.4f}")
print(f"  FPR                : {fpr:.4f}")
print(f"  CV threshold       : {best_thr:.4f}")
print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")

# ── Direct comparison table ────────────────────────────────────────────────────
print(f"\n  {'─'*60}")
print(f"  HEAD-TO-HEAD COMPARISON (identical evaluation conditions)")
print(f"  {'─'*60}")
print(f"  {'Metric':<22} {'XGBoost':>10} {'ConvLSTM':>10} {'Delta':>10}")
print(f"  {'─'*54}")
for metric, xgb_val, conv_val in [
    ('ROC-AUC',   0.929, auc),
    ('Recall',    0.773, rec),
    ('FPR',       0.099, fpr),
    ('Bal. Acc',  0.837, bal),
    ('Precision', 0.004, prec),   # update with your actual XGBoost precision
]:
    delta = conv_val - xgb_val
    arrow = '↑' if delta > 0 else '↓'
    print(f"  {metric:<22} {xgb_val:>10.4f} "
          f"{conv_val:>10.4f}  {arrow}{abs(delta):.4f}")

print(f"\n  Note: Both models evaluated on Ontario pixels only,")
print(f"  fire season DOY {DOY_START_HARD}–{DOY_END_HARD}, "
      f"fire_cause 0–5, val years {VAL_YEARS}")

# ── Save bundle ────────────────────────────────────────────────────────────────
joblib.dump({
    'cv_threshold'   : best_thr,
    'val_auc'        : auc,
    'val_recall'     : rec,
    'val_fpr'        : fpr,
    'val_bal_acc'    : bal,
    'val_precision'  : prec,
    'seq_len'        : SEQ_LEN,
    'H'              : H, 'W': W,
    'n_features'     : N_FEATURES,
    'feat_mean'      : feat_mean,
    'feat_std'       : feat_std,
    'dynamic_bands'  : DYNAMIC_BANDS,
    'static_bands'   : STATIC_BANDS,
    'pos_weight'     : POS_WEIGHT,
    'doy_start'      : DOY_START_HARD,
    'doy_end'        : DOY_END_HARD,
    'val_years'      : VAL_YEARS,
    'train_years'    : TRAIN_YEARS,
}, os.path.join(MODELS_DIR, 'model_convlstm_bundle.pkl'))
print(f"\n  Bundle saved ✓")
print(f"  ROC-AUC: {auc:.4f}  — "
      f"{'beats XGBoost ✓' if auc > 0.929 else 'below XGBoost baseline'}")